# Khmer TTS - Step 3: Train the VITS model

This notebook trains a single-speaker Khmer VITS model using the data from Step 1 (text) and Step 2 (audio).

**Steps**
1. Setup and settings
2. Load the data and check it
3. Build the VITS configuration
4. Train (stops early when the model stops improving)
5. Look at the loss curve
6. Generate Khmer speech samples

**How to run:** Colab menu > Runtime > Change runtime type > **GPU**, then run the cells from top to bottom.
If Colab disconnects, reconnect and run all cells again. Training continues from the last saved checkpoint.

## 1. Setup and settings

In [ ]:
# Connect Google Drive and install the library
from google.colab import drive
drive.mount('/content/drive')

!pip install -q "coqui-tts[codec]==0.28.0" tensorboard pandas soundfile

If pip asks you to restart, use `Runtime > Restart session` once, then continue from the next cell.

In [ ]:
import csv, json, os, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import soundfile as sf
import torch

assert torch.cuda.is_available(), 'Turn on the GPU: Runtime > Change runtime type > GPU'

# ---------- Folders on Google Drive ----------
PROJECT = Path('/content/drive/MyDrive/NLPfinalproject')
PROCESSED = PROJECT / 'processed'  # data from Step 1 and Step 2

# If the folder is not at that path (for example it is only "Shared with me"), look for it in My Drive
if not (PROCESSED / 'characters.txt').exists():
    for folder, subfolders, files in os.walk('/content/drive/MyDrive'):
        subfolders[:] = [s for s in subfolders if s != 'wavs' and folder.count('/') < 8]  # do not walk into the wav folders
        if 'characters.txt' in files:
            PROCESSED = Path(folder)
            PROJECT = PROCESSED.parent
            break
    else:
        print('!! characters.txt was not found anywhere in your My Drive.')
        print('   Fix: open Drive > Shared with me > right-click NLPfinalproject > Organize > Add shortcut > My Drive, then run again.')

OUTPUT = PROJECT / 'training_vits'  # everything from this notebook is saved here
RUNS_DIR = OUTPUT / 'runs'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Training settings (change them here) ----------
MAX_EPOCHS = 50        # never train more than this many epochs
MIN_EPOCHS = 20        # never stop before this many epochs
PATIENCE = 5           # stop if the loss did not improve for this many epochs in a row
BATCH_SIZE = 8         # if you get "CUDA out of memory", change to 8
LEARNING_RATE = 2e-4
MAX_AUDIO_SEC = 12     # longer clips are skipped (they use too much GPU memory)
MIN_AUDIO_SEC = 1
USE_PRETRAINED = True  # start from the English LJSpeech VITS (trains faster). False = start from zero

# Known mistakes in the transcripts: wrong text -> correct text (add more lines when you find more)
# A zero-width space between the words is allowed (\u200b?), so both forms are caught.
CORRECTIONS = {
    'ប្រទេស\u200b?សកម្ពុជា': 'ប្រទេសកម្ពុជា',
}

RUN_NAME = 'vits_khmer'  # change this name to start a brand new training run

## 2. Load the data and check it
- Reads the three metadata files made in Step 1/2 (`id | text | normalized text`)
- Copies the wav files to Colab's local disk (training from Drive directly is very slow)
- Skips clips that are missing, unreadable, or too short / too long

In [ ]:
# Make sure all needed files exist
print('Data folder:', PROCESSED)
print('It contains:', sorted(p.name for p in PROCESSED.iterdir())[:20] if PROCESSED.exists() else 'FOLDER NOT FOUND')
for name in ['characters.txt', 'metadata_train.csv', 'metadata_val.csv', 'metadata_test.csv']:
    assert (PROCESSED / name).is_file(), f'Missing file in processed/: {name}'
assert (PROCESSED / 'wavs').is_dir(), 'Missing folder: processed/wavs'

# Local folder for fast training
LOCAL = Path('/content/khmer_data')
(LOCAL / 'wavs').mkdir(parents=True, exist_ok=True)


def read_metadata(split):
    """Read one metadata file and return a table with the columns: id, text"""
    table = pd.read_csv(PROCESSED / f'metadata_{split}.csv', sep='|', header=None,
                        names=['id', 'text', 'text_normalized'], dtype=str,
                        keep_default_na=False, quoting=csv.QUOTE_NONE, encoding='utf-8-sig')
    table['id'] = table['id'].str.strip().str.replace('.wav', '', regex=False)
    # use the normalized text; if it is empty use the normal text
    table['text'] = table['text_normalized'].where(table['text_normalized'] != '', table['text'])
    table['text'] = table['text'].str.replace('|', ' ', regex=False).str.strip()

    # fix known transcript mistakes
    for wrong, right in CORRECTIONS.items():
        fixed = table['text'].str.replace(wrong, right, regex=True)
        print(f'  {split}: corrected {int((fixed != table["text"]).sum())} sentences ({right})')
        table['text'] = fixed
    return table[['id', 'text']][table['text'] != '']


data = {split: read_metadata(split) for split in ['train', 'val', 'test']}
for split in data:
    print(split, ':', len(data[split]), 'rows in the metadata')


def copy_wav(wav_id):
    source = PROCESSED / 'wavs' / f'{wav_id}.wav'
    target = LOCAL / 'wavs' / f'{wav_id}.wav'
    if target.exists():
        return True
    if source.exists():
        shutil.copyfile(source, target)
        return True
    return False


# Fastest way: if processed/ has ONE zip file with all wavs (wavs.zip), copy that single file and unzip it locally
import zipfile
from tqdm.auto import tqdm

zip_on_drive = PROCESSED / 'wavs.zip'
if zip_on_drive.exists() and not any((LOCAL / 'wavs').glob('*.wav')):
    print('Found wavs.zip - copying one big file (much faster than thousands of small files)...')
    shutil.copyfile(zip_on_drive, LOCAL / 'wavs.zip')
    with zipfile.ZipFile(LOCAL / 'wavs.zip') as archive:
        for name in tqdm(archive.namelist(), desc='Unzipping'):
            if name.lower().endswith('.wav'):
                (LOCAL / 'wavs' / Path(name).name).write_bytes(archive.read(name))

# Copy the wav files that are not on the local disk yet (32 at a time, with a progress bar)
all_ids = sorted(set(pd.concat(data.values())['id']))
with ThreadPoolExecutor(32) as pool:
    copied = dict(zip(all_ids, tqdm(pool.map(copy_wav, all_ids), total=len(all_ids), desc='Copying wavs')))
print('Missing wav files:', sum(1 for ok in copied.values() if not ok))


def get_audio_info(wav_id):
    """Return (sample rate, duration in seconds) or None if the file cannot be read"""
    try:
        info = sf.info(LOCAL / 'wavs' / f'{wav_id}.wav')
        return info.samplerate, info.duration
    except Exception:
        return None


audio_info = {i: get_audio_info(i) for i in all_ids if copied[i]}

# Sample rate = the most common one in the data
rates = pd.Series([v[0] for v in audio_info.values() if v]).value_counts()
SAMPLE_RATE = int(rates.index[0])
print('Sample rates found:', rates.to_dict(), '-> using', SAMPLE_RATE, 'Hz')

# Keep only good clips and save the metadata in the format the trainer needs
summary = []
for split in data:
    table = data[split]
    keep = []
    for wav_id in table['id']:
        info = audio_info.get(wav_id)
        keep.append(bool(info) and info[0] == SAMPLE_RATE and MIN_AUDIO_SEC <= info[1] <= MAX_AUDIO_SEC)
    removed = len(table) - sum(keep)
    table = table[keep].reset_index(drop=True)
    hours = sum(audio_info[i][1] for i in table['id']) / 3600
    data[split] = table
    summary.append({'split': split, 'used': len(table), 'removed': removed, 'hours': round(hours, 2)})

    with open(LOCAL / f'metadata_{split}.csv', 'w', encoding='utf-8') as f:
        for wav_id, text in zip(table['id'], table['text']):
            f.write(f'{wav_id}|{text}|{text}\n')

print(pd.DataFrame(summary).to_string(index=False))
assert len(data['train']) > 1000, 'Too few training clips. Check the output of Step 1 and Step 2.'
assert len(data['val']) > 20, 'Too few validation clips.'

## 3. Build the VITS configuration
- The Khmer character list comes from `characters.txt` (Step 1).
- One speaker only, so no speaker embedding.
- Mixed precision is off because VITS can become unstable with it.
- `target_loss = 'loss_mel'`: the best model and early stopping are based on the mel-spectrogram loss.
 (The default loss mixes generator and discriminator, which work against each other, so it is not a good "better or worse" signal.)

In [ ]:
from TTS.config.shared_configs import BaseAudioConfig, BaseDatasetConfig
from TTS.tts.configs.shared_configs import CharactersConfig
from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.vits import Vits, VitsArgs
from TTS.tts.utils.text.tokenizer import TTSTokenizer

# ---- Characters ----
PUNCTUATION = " !',.?។"
text_in_file = (PROCESSED / 'characters.txt').read_text(encoding='utf-8')
CHARS = ''.join(c for c in dict.fromkeys(text_in_file) if c not in PUNCTUATION + '\\\n\r\t')
KNOWN = set(CHARS) | set(PUNCTUATION)
print('Number of Khmer characters:', len(CHARS))

# Tell us if a sentence has characters that the model does not know (they would be dropped)
for split in data:
    unknown = data[split]['text'].map(lambda t: [c for c in t if c not in KNOWN])
    count = int((unknown.map(len) > 0).sum())
    if count:
        print(f'{split}: {count} sentences have unknown characters, for example', unknown[unknown.map(len) > 0].iloc[0])

# ---- Configuration ----
audio_config = BaseAudioConfig(sample_rate=SAMPLE_RATE, win_length=1024, hop_length=256,
                               num_mels=80, mel_fmin=0, mel_fmax=None)

characters_config = CharactersConfig(
    characters_class='TTS.tts.utils.text.characters.Graphemes',
    pad='<PAD>', eos='<EOS>', bos='<BOS>', blank='<BLNK>',
    characters=CHARS, punctuations=PUNCTUATION,
)

dataset_config = BaseDatasetConfig(
    formatter='ljspeech', dataset_name='khmer', path=str(LOCAL),
    meta_file_train='metadata_train.csv', meta_file_val='metadata_val.csv', language='km',
)

config = VitsConfig(
    model_args=VitsArgs(use_speaker_embedding=False),
    audio=audio_config,
    characters=characters_config,
    datasets=[dataset_config],
    run_name=RUN_NAME,
    output_path=str(RUNS_DIR),
    # training
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    eval_batch_size=min(BATCH_SIZE, 16),
    lr_gen=LEARNING_RATE,
    lr_disc=LEARNING_RATE,
    mixed_precision=False,
    num_loader_workers=4,
    num_eval_loader_workers=2,
    # text and audio
    text_cleaner='basic_cleaners',
    use_phonemes=False,
    add_blank=True,
    min_text_len=3,
    max_text_len=255,
    min_audio_len=int(SAMPLE_RATE * MIN_AUDIO_SEC),
    max_audio_len=int(SAMPLE_RATE * MAX_AUDIO_SEC),
    # evaluation, saving and logging
    run_eval=True,
    target_loss='loss_mel',
    test_delay_epochs=-1,
    test_sentences=list(data['val']['text'].iloc[:2]),  # read out during training (see TensorBoard)
    save_step=2000,
    save_n_checkpoints=2,
    save_best_after=0,
    print_step=100,
    plot_step=500,
    log_model_step=5000,
    print_eval=False,
)

train_samples, eval_samples = load_tts_samples(config, eval_split=True)
steps_per_epoch = len(train_samples) // BATCH_SIZE
print('Training samples  :', len(train_samples))
print('Validation samples:', len(eval_samples))
print(f'About {steps_per_epoch} steps per epoch, {steps_per_epoch * MAX_EPOCHS:,} steps in {MAX_EPOCHS} epochs')

## 4. Train with early stopping
After every epoch the validation mel loss is saved in a small json file in Drive.
- Before `MIN_EPOCHS` we never stop.
- After that we stop when the loss did not get better for `PATIENCE` epochs in a row.
- We always stop at `MAX_EPOCHS`.

The model with the lowest validation loss is saved as `best_model.pth`.

**Pretrained start:** we begin from the English LJSpeech VITS so the model learns faster. Only the Khmer text part is learned from zero.

In [ ]:
from trainer import Trainer, TrainerArgs

HISTORY_FILE = OUTPUT / f'{RUN_NAME}_val_loss.json'  # validation loss of every epoch

# Runs that were already started (so we can continue after a disconnect)
old_runs = sorted(RUNS_DIR.glob(f'{RUN_NAME}-*'), key=lambda p: p.stat().st_mtime)
old_runs = [run for run in old_runs if list(run.glob('*.pth'))]
if not old_runs and HISTORY_FILE.exists():
    HISTORY_FILE.unlink()  # old history without a model: start fresh


def load_history():
    if HISTORY_FILE.exists():
        return {int(epoch): loss for epoch, loss in json.loads(HISTORY_FILE.read_text()).items()}
    return {}


def check_stop(history):
    """Decide if training should stop. Returns (True/False, message)."""
    if not history:
        return False, ''
    epochs_done = max(history) + 1
    if epochs_done >= MAX_EPOCHS:
        return True, f'Reached the maximum of {MAX_EPOCHS} epochs.'
    if epochs_done < MIN_EPOCHS:
        return False, ''

    # find the best epoch (a gain smaller than 0.5% does not count)
    best_loss, best_epoch = float('inf'), 0
    for epoch in sorted(history):
        if history[epoch] < best_loss * 0.995:
            best_loss, best_epoch = history[epoch], epoch

    epochs_without_gain = max(history) - best_epoch
    if epochs_without_gain >= PATIENCE:
        return True, f'Early stop: no improvement for {epochs_without_gain} epochs (best epoch {best_epoch + 1}, loss {best_loss:.4f}).'
    return False, ''


class StopTraining(Exception):
    """Raised to stop the training on purpose"""


history = load_history()


def on_epoch_end(trainer):
    """Called by the trainer after every epoch"""
    loss = trainer._pick_target_avg_loss(trainer.keep_avg_eval)  # validation mel loss
    if loss is None:
        return
    history[trainer.epochs_done] = float(loss)
    HISTORY_FILE.write_text(json.dumps(history, indent=2))
    print(f'>>> Epoch {trainer.epochs_done + 1}/{MAX_EPOCHS} | validation mel loss = {loss:.4f}')

    stop, message = check_stop(history)
    if stop:
        print('>>>', message)
        raise StopTraining()


class EarlyStopTrainer(Trainer):
    """The normal Trainer, but it can stop cleanly when StopTraining is raised"""
    def fit(self):
        try:
            self._fit()
        except StopTraining:
            self.save_checkpoint()
            self.dashboard_logger.finish()


def prepare_pretrained(path, model):
    """Make the English checkpoint usable for the Khmer model.
    New PyTorch names some weights differently ('weight_g' -> 'parametrizations.weight.original0').
    We rename them, otherwise about 40% of the pretrained layers would not be loaded."""
    checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    new_names = any('parametrizations.weight.original0' in name for name in model.state_dict())
    weights = {}
    for name, value in checkpoint['model'].items():
        if new_names and name.endswith('.weight_g'):
            name = name.replace('.weight_g', '.parametrizations.weight.original0')
        elif new_names and name.endswith('.weight_v'):
            name = name.replace('.weight_v', '.parametrizations.weight.original1')
        weights[name] = value
    checkpoint['model'] = weights
    for key in ['optimizer', 'scheduler', 'scaler']:  # start with fresh optimizers for Khmer
        checkpoint.pop(key, None)
    new_path = LOCAL / 'pretrained_english_vits.pth'
    torch.save(checkpoint, new_path)
    return str(new_path)


# ---------------- Start training ----------------
finished, message = check_stop(history)
if finished:
    print('Training is already finished.', message)
else:
    tokenizer, config = TTSTokenizer.init_from_config(config)
    model = Vits(config, tokenizer=tokenizer)

    continue_path = ''  # continue an old run
    restore_path = ''   # or start from the pretrained English model
    if old_runs:
        continue_path = str(old_runs[-1])
        print('Continuing from:', continue_path)
    elif USE_PRETRAINED:
        try:
            from TTS.utils.manage import ModelManager
            english_model, _, _ = ModelManager().download_model('tts_models/en/ljspeech/vits')
            restore_path = prepare_pretrained(english_model, model)
            print('Starting from the pretrained English VITS')
        except Exception as error:
            print('Could not use the pretrained model, starting from zero instead:', repr(error))
    else:
        print('Starting from zero (USE_PRETRAINED = False)')

    trainer = EarlyStopTrainer(
        TrainerArgs(continue_path=continue_path, restore_path=restore_path),
        config,
        output_path=str(RUNS_DIR),
        model=model,
        train_samples=train_samples,
        eval_samples=eval_samples,
        callbacks={'on_epoch_end': on_epoch_end},
    )
    trainer.fit()
    print('Training finished.')

## 5. Loss curve and summary

In [ ]:
import matplotlib.pyplot as plt

history = load_history()
assert history, 'No history found. Run the training cell first.'

epochs = sorted(history)
losses = [history[e] for e in epochs]
best_epoch = epochs[losses.index(min(losses))]

plt.figure(figsize=(8, 4))
plt.plot([e + 1 for e in epochs], losses, marker='o')
plt.axvline(best_epoch + 1, color='green', linestyle='--', label=f'best epoch: {best_epoch + 1}')
plt.xlabel('Epoch')
plt.ylabel('Validation mel loss')
plt.title('VITS Khmer - validation loss')
plt.legend()
plt.grid(True)
plt.savefig(OUTPUT / 'loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()

summary = {
 'epochs_trained': len(epochs),
 'max_epochs': MAX_EPOCHS,
 'best_epoch': best_epoch + 1,
 'best_validation_loss': min(losses),
 'stopped_early': len(epochs) < MAX_EPOCHS,
 'pretrained_start': USE_PRETRAINED,
 'batch_size': BATCH_SIZE,
 'learning_rate': LEARNING_RATE,
 'sample_rate': SAMPLE_RATE,
 'train_clips': len(data['train']),
 'val_clips': len(data['val']),
 'test_clips': len(data['test']),
}
(OUTPUT / 'training_summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

## 6. Generate speech samples
- **In-domain:** sentences from the test set (same topics as the training data). The original recording is saved next to each one.
- **Out-of-domain:** new Khmer sentences that are not in the training data.

The wav files and `samples_manifest.csv` are saved in `training_vits/samples` for Member 4 (CER and MOS).

In [ ]:
from IPython.display import Audio, display
from TTS.utils.synthesizer import Synthesizer

# ---- Choose the run folder and the best model ----
# Normally the latest run. To use a specific run, write its folder name here, for example:
# RUN_FOLDER = 'vits_khmer-October-27-2026_04+36AM-0000000'
RUN_FOLDER = None

if RUN_FOLDER:
    run_dir = RUNS_DIR / RUN_FOLDER
else:
    run_dir = sorted(RUNS_DIR.glob(f'{RUN_NAME}-*'), key=lambda p: p.stat().st_mtime)[-1]

# best_model_<step>.pth with the highest step (files smaller than 1 MB are broken, skip them)
best_models = [p for p in run_dir.glob('best_model*.pth') if p.stat().st_size > 1_000_000]
best_models = sorted(best_models, key=lambda p: p.stat().st_mtime)
assert best_models, 'No valid best_model*.pth found in ' + str(run_dir)
checkpoint = best_models[-1]
print('Using model:', checkpoint.name, 'from', run_dir.name)


def load_synthesizer(run_dir, checkpoint):
    """config.json and the checkpoint can disagree about the character class
    (error: size mismatch for text_encoder.emb.weight). We try both classes and keep the one that fits."""
    saved = json.loads((run_dir / 'config.json').read_text(encoding='utf-8'))
    classes = ['TTS.tts.utils.text.characters.Graphemes', 'TTS.tts.models.vits.VitsCharacters']
    classes.sort(key=lambda c: c != saved['characters']['characters_class'])  # try the saved one first
    for character_class in classes:
        saved['characters']['characters_class'] = character_class
        fixed_config = run_dir / 'config_fixed.json'
        fixed_config.write_text(json.dumps(saved, ensure_ascii=False), encoding='utf-8')
        try:
            synth = Synthesizer(tts_checkpoint=str(checkpoint), tts_config_path=str(fixed_config), use_cuda=torch.cuda.is_available())
            print('Model loaded with character class:', character_class.split('.')[-1])
            return synth
        except RuntimeError as error:
            print('Does not fit:', character_class.split('.')[-1], '-', str(error).splitlines()[-1][:120])
    raise RuntimeError('The model does not match characters.txt. Check that characters.txt is the same file used for training.')


synth = load_synthesizer(run_dir, checkpoint)

# Noise scale used for ALL samples (0.5 sounded best; default is 0.667, lower = steadier pronunciation)
NOISE_SCALE = 0.5
synth.tts_model.inference_noise_scale = NOISE_SCALE

# Sentences
# In-domain: one random sentence from EACH topic of the test set (the model never trained on them).
# Change SAMPLE_SEED to get a different set. Very short and very long sentences are skipped.
SAMPLE_SEED = 1
test_all = data['test'].copy()
test_all['topic'] = test_all['id'].str.replace('\\', '', regex=False).str.extract(r'^(.*?)_\d+_\d+_\d+$')[0].fillna('all')
medium = test_all[test_all['text'].str.len().between(60, 150)]
if len(medium) < 8:
    medium = test_all  # not enough medium sentences: use all
test_table = medium.groupby('topic', group_keys=False).sample(1, random_state=SAMPLE_SEED)
if len(test_table) < 8:  # data without topics: just take 8 random ones
    test_table = medium.sample(min(8, len(medium)), random_state=SAMPLE_SEED)
print('In-domain sentences per topic:'); print(test_table[['topic']].value_counts().to_string())
in_domain = list(zip(test_table['id'], test_table['text']))
out_of_domain = [
    ('ood01', 'ថ្ងៃនេះអាកាសធាតុក្តៅខ្លាំងណាស់។'),
    ('ood02', 'ខ្ញុំចង់ទិញទូរស័ព្ទថ្មីមួយនៅផ្សារ។'),
    ('ood03', 'សូមអរគុណច្រើនចំពោះជំនួយរបស់អ្នក។'),
    ('ood04', 'តើម៉ោងប៉ុន្មានហើយឥឡូវនេះ?'),
    ('ood05', 'រដ្ឋាភិបាលបានប្រកាសគម្រោងថ្មីអំពីការអភិវឌ្ឍបច្ចេកវិទ្យានៅកម្ពុជា។'),
    ('ood06', 'តើអ្នកអាចប្រាប់ខ្ញុំពីរបៀបទៅកាន់សារមន្ទីរជាតិបានទេ?'),
]

samples_dir = OUTPUT / 'samples' / run_dir.name
rows = []

for domain, sentences in [('in_domain', in_domain), ('out_of_domain', out_of_domain)]:
    folder = samples_dir / domain
    folder.mkdir(parents=True, exist_ok=True)

    for number, (sentence_id, text) in enumerate(sentences, 1):
        text = ''.join(c for c in text if c in KNOWN).strip()  # keep only characters the model knows
        if not text:
            continue

        wav = synth.tts(text, split_sentences=False)
        wav_path = folder / f'{domain}_{number:02d}.wav'
        synth.save_wav(wav, str(wav_path))

        if domain == 'in_domain':  # save the real recording too, to compare
            shutil.copyfile(LOCAL / 'wavs' / f'{sentence_id}.wav', folder / f'{domain}_{number:02d}_original.wav')

        rows.append({'domain': domain, 'id': sentence_id, 'text': text, 'wav': str(wav_path),
                     'seconds': round(len(wav) / synth.output_sample_rate, 2)})
        print(domain, number, ':', text)
        display(Audio(str(wav_path)))

pd.DataFrame(rows).to_csv(samples_dir / 'samples_manifest.csv', index=False, encoding='utf-8-sig')
print('Saved', len(rows), 'samples in:', samples_dir)

## (Optional) TensorBoard
Shows all detailed loss curves (generator, discriminator, mel, KL, duration) and the speech made during training.

In [ ]:
# Open TensorBoard on the training logs
%load_ext tensorboard
%tensorboard --logdir {str(OUTPUT)}